# Práctica 2 · Minería de procesos: ¿cómo se atiende *realmente* un incidente?
**Analítica de procesos para ciberseguridad**

**El problema.** Todo equipo tiene un procedimiento escrito: *«un incidente se registra, se investiga, se resuelve y se cierra»*. Pero lo que **de verdad ocurre** casi nunca es idéntico a lo escrito: hay incidentes que se saltan pasos, que esperan semanas por alguien más, que rebotan entre equipos. La **minería de procesos** reconstruye el proceso real **a partir de los registros que dejan los sistemas**, y lo compara con el esperado.

**Tres ideas para leer este cuaderno:**

| Idea | Qué es | En nuestros datos |
|---|---|---|
| **Caso** | Una «instancia» del proceso: lo que se sigue de principio a fin | Un incidente |
| **Actividad** | Un paso que ocurre en el caso | Un cambio de estado: *Nuevo*, *En curso*, *Resuelto*... |
| **Evento** | Un paso concreto de un caso, con **fecha y hora** | «El incidente INC0000045 pasó a *Resuelto* el 29/02 a las 08:53» |

Con esos tres datos (caso, actividad, fecha) se puede hacer **tres cosas**: **descubrir** el proceso real, **comprobar la conformidad** (¿se siguió el procedimiento?) y **mejorar** (¿dónde se pierde tiempo?). Las tres están en este cuaderno.

**Los datos.** El registro de incidentes de una **mesa de servicio de TI** (plataforma ServiceNow de una empresa): 24.916 incidentes y más de 141.000 eventos. No es un SOC de seguridad, pero un incidente de seguridad se gestiona igual: se abre, se asigna, se investiga, se espera información, se resuelve y se cierra. Los datos personales ya vienen anonimizados.

> **Fuente y cambios.** Amaral, Fantinato, Reijers y Peres (2018). *Incident management process enriched event log.* UCI Machine Learning Repository, https://doi.org/10.24432/C57S4H. Licencia CC BY 4.0. Cambios hechos con fines académicos: se quitaron 2 incidentes con un estado inválido, se tradujeron al español los estados y otras etiquetas, y las fechas pasaron al formato AAAA-MM-DD (script `scripts/preparar_servicenow.py`).

## 0 · Preparación

In [ ]:
# Importa las librerías: pandas (tablas), numpy (cálculos) y matplotlib (gráficos)
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")                # oculta advertencias para que la salida se lea limpia
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# Dirección de los datos: repositorio de GitHub y versión fija de donde se descargan los archivos
REPO = "lsandinop/analitica-procesos-ciberseguridad"
VERSION = "main"
DATA_BASE = "" if REPO.startswith("USUARIO") else f"https://raw.githubusercontent.com/{REPO}/{VERSION}/datos/"

def cargar(nombre, fechas=None):
    # Lee un archivo CSV y lo devuelve como tabla (DataFrame). Lo busca primero en GitHub;
    # si no está disponible, en la carpeta local. `fechas` son las columnas que se leen como fecha y hora.
    if DATA_BASE:
        try:
            return pd.read_csv(DATA_BASE + nombre, parse_dates=fechas)
        except Exception:
            print("No pude descargar el archivo de GitHub; busco una copia local.")
    for ruta in (nombre, "datos/" + nombre, "../datos/" + nombre, "../../datos/" + nombre):
        if os.path.exists(ruta):
            return pd.read_csv(ruta, parse_dates=fechas)
    raise FileNotFoundError(f"No encuentro {nombre}: súbelo a la carpeta del cuaderno")

eventos = cargar("incidentes_servicenow.csv", fechas=["fecha_evento", "fecha_apertura", "fecha_resolucion", "fecha_cierre"])
print(f"{len(eventos):,} filas · {eventos.incidente.nunique():,} incidentes".replace(",", "."))

## 1 · Los datos: un incidente es una historia
Cada fila del archivo es un **evento**: algo que cambió en un incidente (de estado, de grupo, de analista) en un momento dado. Estas son las primeras filas:

In [ ]:
eventos[["incidente", "estado", "fecha_evento", "prioridad", "grupo_asignado", "reasignaciones"]].head(10)

**Un solo incidente, como línea de tiempo.** Así se ve la historia de un caso. Miremos uno con una decena de filas:

In [ ]:
def linea_de_tiempo(id_incidente):
    # Imprime los cambios de estado de un incidente, con cuánto tiempo pasó desde que se abrió
    e = eventos[eventos.incidente == id_incidente].sort_values("fecha_evento", kind="stable")
    apertura = e.fecha_apertura.iloc[0]
    print(f"Incidente {id_incidente} · prioridad {e.prioridad.iloc[0]} · abierto el {apertura:%Y-%m-%d %H:%M}\n")
    anterior = None
    for _, f in e.iterrows():
        if f.estado != anterior:                                    # solo cuando cambia el estado
            horas = (f.fecha_evento - apertura).total_seconds() / 3600
            print(f"  +{horas:7.1f} h   {f.estado}")
            anterior = f.estado

filas_por_incidente = eventos.groupby("incidente").size()
ejemplo = filas_por_incidente[filas_por_incidente.between(8, 12)].index[0]       # un incidente con una decena de filas
linea_de_tiempo(ejemplo)

## 2 · De las filas al registro de eventos
En el archivo hay **muchas más filas que cambios de estado**: cada vez que alguien edita *cualquier* campo se guarda una fila nueva, aunque el estado siga igual. Para estudiar el **proceso** solo nos interesan los **cambios de estado**, así que dejamos una fila por cambio:

In [ ]:
eventos = eventos.sort_values(["incidente", "fecha_evento"], kind="stable")

# Un cambio de estado = una fila cuyo estado es distinto del de la fila anterior del MISMO incidente
cambio = eventos.estado != eventos.groupby("incidente").estado.shift()
registro = eventos.loc[cambio, ["incidente", "estado", "fecha_evento"]].reset_index(drop=True)

print(f"Filas en el archivo: {len(eventos):,} · cambios de estado (eventos del proceso): {len(registro):,}".replace(",", "."))
print(f"Eventos por incidente: mediana {registro.groupby('incidente').size().median():.0f}, máximo {registro.groupby('incidente').size().max()}")
registro.head(8)

## 3 · Variantes: los caminos que siguen los incidentes
Una **variante** es un camino completo, de principio a fin. Dos incidentes con la misma secuencia de estados son la misma variante. Contémoslas:

In [ ]:
# La ruta de cada incidente: sus estados, en orden, unidos con flechas
rutas = registro.groupby("incidente").estado.apply(lambda s: " → ".join(s))
variantes = rutas.value_counts().rename("incidentes").to_frame()
variantes["% del total"] = variantes.incidentes / variantes.incidentes.sum() * 100
variantes["% acumulado"] = variantes["% del total"].cumsum()
print(f"{len(variantes)} variantes distintas en {len(rutas):,} incidentes".replace(",", "."))
for objetivo in (50, 80, 90):
    print(f"  {int((variantes['% acumulado'] < objetivo).sum() + 1)} variantes cubren el {objetivo} % de los incidentes")
variantes.head(10).round(1)

In [ ]:
# Gráfico de Pareto: cuántas variantes hacen falta para cubrir los incidentes
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(range(1, len(variantes) + 1), variantes["% acumulado"], "o-", color="#1f6db0", markersize=3)
ax.axhline(80, color="#d9534f", ls="--", lw=1); ax.text(len(variantes) * 0.55, 82, "80 % de los incidentes", color="#d9534f")
ax.set_xlabel("número de variantes (de la más común a la menos común)"); ax.set_ylabel("% acumulado de incidentes")
ax.set_title("Unas pocas rutas concentran casi todos los incidentes"); plt.tight_layout(); plt.show()

**Qué leer ahí.** Hay 188 caminos distintos, pero **5 variantes concentran el 80 %** de los incidentes. El resto es una «cola larga» de casos raros. Eso es típico de los procesos reales: **pocas rutas comunes y muchas excepciones**.

Y fíjate en la ruta más común: *Nuevo → Resuelto → Cerrado*. **Se salta «En curso».** El procedimiento dice que un incidente se investiga antes de resolverse, y la cuarta parte de los incidentes no pasa por ese estado. Lo veremos otra vez más abajo.

## 4 · El mapa del proceso
Un **mapa de proceso** dibuja los estados como cajas y los **traspasos** entre ellos como flechas. Cuanto más gruesa la flecha, más incidentes pasaron por ahí; el número es la cantidad. Primero contamos los traspasos:

In [ ]:
# Traspasos: de qué estado a cuál pasa cada incidente (incluyendo el inicio y el fin del proceso)
registro["siguiente"] = registro.groupby("incidente").estado.shift(-1)
registro["siguiente"] = registro.siguiente.fillna("Fin")                       # tras el último estado, el proceso termina
primeros = registro.groupby("incidente").estado.first()
traspasos = registro.groupby(["estado", "siguiente"]).size().to_dict()
traspasos.update({("Inicio", e): n for e, n in primeros.value_counts().items()})
n_incidentes = registro.incidente.nunique()
print(f"{len(traspasos)} traspasos distintos (con inicio y fin). Los 6 más frecuentes entre estados:")
reales = {k: v for k, v in traspasos.items() if k[0] != "Inicio" and k[1] != "Fin"}          # sin los marcadores de inicio y fin
for (a, b), n in sorted(reales.items(), key=lambda x: -x[1])[:6]:
    print(f"   {a:22s} → {b:22s} {n:6,}".replace(",", "."))

**Dibujamos el mapa.** La función coloca cada estado en una posición fija (de izquierda a derecha, en el orden del procedimiento) y dibuja las flechas. Con `min_pct` se oculta lo poco frecuente:

In [ ]:
POSICION = {"Inicio": (0, 0), "Nuevo": (1.5, 0), "En curso": (3.2, 0), "Esperando al usuario": (4.9, 1.5), "Esperando al proveedor": (4.9, 0.5),
            "Esperando análisis del problema": (4.9, -0.5), "Esperando evidencia": (4.9, -1.5), "Resuelto": (6.7, 0), "Cerrado": (8.1, 0), "Fin": (9.2, 0)}
ROTULO = {"Esperando al usuario": "Esperando\nal usuario", "Esperando al proveedor": "Esperando\nal proveedor", "Esperando análisis del problema": "Esperando\nanálisis problema",
          "Esperando evidencia": "Esperando\nevidencia", "En curso": "En curso"}

# Altura del arco (en unidades del dibujo) para los traspasos que «saltan» por encima de otras cajas. Positivo = arco hacia arriba.
ARCO = {("Inicio", "En curso"): 0.9, ("Inicio", "Resuelto"): 2.0, ("Nuevo", "Resuelto"): 1.4, ("En curso", "Resuelto"): -0.9, ("Nuevo", "Cerrado"): -1.9, ("En curso", "Cerrado"): -1.5, ("Resuelto", "En curso"): -1.8}

def dibujar_mapa(traspasos, titulo, min_pct=0.0, rotular_desde_pct=1.0):
    # Dibuja el mapa del proceso: cajas = estados, flechas = traspasos (grosor proporcional a la cantidad de incidentes).
    # Solo se escribe el número en los traspasos que superan `rotular_desde_pct` % de los incidentes, para que se pueda leer.
    fig, ax = plt.subplots(figsize=(13, 5.8))
    visibles = {k: v for k, v in traspasos.items() if v / n_incidentes * 100 >= min_pct}
    maximo = max(visibles.values())
    for (a, b), n in sorted(visibles.items(), key=lambda x: x[1]):
        (x1, y1), (x2, y2) = POSICION[a], POSICION[b]
        retroceso = x2 < x1                                                     # el incidente vuelve a un estado anterior
        altura = ARCO.get((a, b), -0.8 if retroceso else 0.0)
        if x2 == x1:                                                            # entre dos «esperas» (misma columna): curva leve para que no se tapen
            rad, altura = 0.3, 0.0
        else:
            rad = -2 * altura / (x2 - x1) if altura else 0.0                    # matplotlib describe la curva con «rad»; esto la convierte desde la altura
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", lw=0.7 + 8 * n / maximo, color="#d9534f" if retroceso else "#1f6db0",
                    alpha=0.7, shrinkA=26, shrinkB=26, connectionstyle=f"arc3,rad={rad}"))
        if n / n_incidentes * 100 >= rotular_desde_pct:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2 + altura + (0.18 if altura >= 0 else -0.22), f"{n:,}".replace(",", "."), ha="center", fontsize=8, color="#14212b",
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9), zorder=4)
    usados = {a for a, _ in visibles} | {b for _, b in visibles}
    for nombre, (x, y) in POSICION.items():
        if nombre in usados:
            marco = dict(boxstyle="round,pad=0.45", fc="#0f2437" if nombre in ("Inicio", "Fin") else "#eaf1f8", ec="#0f2437")
            ax.text(x, y, ROTULO.get(nombre, nombre), ha="center", va="center", fontsize=9, color="white" if nombre in ("Inicio", "Fin") else "#0f2437", bbox=marco, zorder=5)
    ax.set_xlim(-0.8, 9.9); ax.set_ylim(-2.6, 3.0); ax.axis("off")
    ax.set_title(f"{titulo}  ·  {len(visibles)} traspasos (rojo: el incidente retrocede)", fontsize=11); plt.tight_layout(); plt.show()

dibujar_mapa(traspasos, "Mapa completo del proceso")

In [ ]:
# El mismo proceso, mostrando solo los traspasos que ocurren en al menos el 1 % de los incidentes
dibujar_mapa(traspasos, "Mapa simplificado (1 % o más)", min_pct=1.0)

**Qué leer ahí.** El mapa completo es difícil de leer: demasiadas flechas. El simplificado deja ver lo esencial: el flujo principal de izquierda a derecha, el desvío por las **esperas** y los pocos retrocesos. **Simplificar es una decisión:** ocultar lo raro aclara el proceso, pero también oculta las excepciones, que a veces son justo lo que importa en seguridad.

Hay también una cosa que no se ve en un mapa de frecuencias: **cuánto tarda** cada traspaso.

## 5 · ¿Dónde se pierde el tiempo?
Como cada evento tiene fecha y hora, podemos medir **cuánto pasa entre un estado y el siguiente**. Calculamos, para cada traspaso, cuánto esperó cada incidente:

In [ ]:
registro["fecha_siguiente"] = registro.groupby("incidente").fecha_evento.shift(-1)
registro["horas_en_estado"] = (registro.fecha_siguiente - registro.fecha_evento).dt.total_seconds() / 3600
tiempos = (registro.dropna(subset=["horas_en_estado"]).groupby(["estado", "siguiente"]).horas_en_estado
           .agg(incidentes="size", mediana_horas="median", total_dias=lambda s: s.sum() / 24).sort_values("total_dias", ascending=False))
tiempos["mediana_dias"] = tiempos.mediana_horas / 24
tiempos.head(10).round(1)

In [ ]:
# Mediana de espera en los traspasos más frecuentes y en los más lentos (escala logarítmica: las diferencias son enormes)
frecuentes = tiempos[tiempos.incidentes >= 150].copy()
frecuentes["traspaso"] = [f"{a} → {b}" for a, b in frecuentes.index]
frecuentes = frecuentes.sort_values("mediana_horas").tail(12)
ax = frecuentes.set_index("traspaso").mediana_horas.plot.barh(figsize=(9, 4.6), color="#1f6db0", logx=True)
for i, (v, n) in enumerate(zip(frecuentes.mediana_horas, frecuentes.incidentes)):
    ax.text(v * 1.08, i, f"{v / 24:.1f} días · {n:,} incidentes".replace(",", "."), va="center", fontsize=8)
ax.set_xlabel("mediana de horas hasta el siguiente estado (escala logarítmica)"); ax.set_ylabel("")
ax.set_title("Dónde esperan los incidentes (traspasos con 150 incidentes o más)"); ax.set_xlim(right=frecuentes.mediana_horas.max() * 12); plt.tight_layout(); plt.show()

**Qué leer ahí.**
- ***Resuelto → Cerrado*** **tarda unos 5 días en casi todos los incidentes.** Eso no es trabajo del equipo: es una espera fija (un cierre automático). Es el traspaso con más «tiempo acumulado» y el que menos se puede mejorar sin cambiar la política.
- Las **esperas por el proveedor** son **raras** (menos de 200 incidentes) pero duran **semanas o meses**.
- Cuando el incidente espera **información del usuario**, la mediana es de varios días.

**Cuidado con «dónde se pierde más tiempo en total»:** un traspaso lento y raro (proveedor) pesa menos en el total que uno corto y masivo. Mirar solo el total o solo la mediana cuenta historias distintas.

**Tiempo total hasta resolver.** Desde que se abre el incidente hasta que se resuelve:

In [ ]:
por_incidente = eventos.groupby("incidente").agg(apertura=("fecha_apertura", "first"), resolucion=("fecha_resolucion", "last"))
dias = (por_incidente.resolucion - por_incidente.apertura).dt.total_seconds() / 86400
print(f"Hasta resolver: mediana {dias.median():.1f} días · el 90 % se resuelve en {dias.quantile(0.9):.1f} días · el más lento tardó {dias.max():.0f} días")
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.hist(dias.clip(lower=0.01), bins=np.logspace(-2, 2.6, 50), color="#1f6db0"); ax.set_xscale("log")
ax.axvline(dias.median(), color="#d9534f", ls="--"); ax.set_xlabel("días desde la apertura hasta la resolución (escala logarítmica)"); ax.set_ylabel("incidentes")
ax.set_title("Tiempo hasta resolver"); plt.tight_layout(); plt.show()

## 6 · Conformidad: ¿se sigue el procedimiento?
El procedimiento escrito de esta mesa de servicio es simple: **Nuevo → En curso → Resuelto → Cerrado**. La **comprobación de conformidad** compara cada incidente real contra ese camino y dice cuáles se desviaron. Usamos una librería profesional de minería de procesos, **PM4Py**, que implementa estas técnicas:

In [ ]:
# Instala PM4Py (la primera vez tarda un momento). Si ya está instalada, no hace nada.
try:
    import pm4py
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pm4py"], check=True)
    import pm4py

# PM4Py necesita saber qué columna es el caso, cuál la actividad y cuál la fecha
log = pm4py.format_dataframe(registro[["incidente", "estado", "fecha_evento"]], case_id="incidente", activity_key="estado", timestamp_key="fecha_evento")

# El procedimiento esperado, escrito como un «árbol de proceso»: -> significa «en este orden»
esperado = pm4py.parse_process_tree("->('Nuevo','En curso','Resuelto','Cerrado')")
red, inicio, fin = pm4py.convert_to_petri_net(esperado)         # PM4Py lo convierte al formato con que compara los incidentes
print("Procedimiento esperado:", esperado)

In [ ]:
# Para cada incidente: ¿encaja con el procedimiento? (y cuánto: 1,00 = perfecto)
diagnostico = pm4py.conformance_diagnostics_token_based_replay(log, red, inicio, fin)
incidentes_ordenados = log.groupby("case:concept:name").size().index
conformidad = pd.DataFrame({"ruta": rutas.reindex(incidentes_ordenados).values, "encaja": [d["trace_is_fit"] for d in diagnostico],
                            "ajuste": [d["trace_fitness"] for d in diagnostico]}, index=incidentes_ordenados)
print(f"{conformidad.encaja.mean():.1%} de los incidentes encaja con el procedimiento esperado ({int(conformidad.encaja.sum()):,} de {len(conformidad):,})".replace(",", "."))
tabla = conformidad.groupby("ruta").agg(incidentes=("encaja", "size"), encaja=("encaja", "mean"), ajuste=("ajuste", "mean")).sort_values("incidentes", ascending=False).head(8)
tabla.round(2)

**Cómo leerlo.** Las rutas que encajan son las que siguen el orden *Nuevo → En curso → Resuelto → Cerrado*. Fíjate en una cosa: **las esperas no están en el procedimiento**, y PM4Py las tolera (no penaliza pasos que el modelo no conoce), por eso *Nuevo → En curso → Esperando al usuario → Resuelto → Cerrado* también encaja.

Las que **no** encajan son las que **se saltan** pasos. Clasifiquémoslas por tipo de desvío:

In [ ]:
def clasificar(ruta):
    # Clasifica la ruta de un incidente respecto del procedimiento esperado
    pasos = ruta.split(" → ")
    sin_esperas = [p for p in pasos if not p.startswith("Esperando")]
    if sin_esperas == ["Nuevo", "En curso", "Resuelto", "Cerrado"]:
        return "1 · Sigue el procedimiento (con o sin esperas)"
    if "En curso" not in pasos:
        return "2 · Se resuelve sin pasar por «En curso»"
    if pasos[0] != "Nuevo":
        return "3 · Empieza en «En curso» (no se registró «Nuevo»)"
    return "4 · Otro desvío (retrocesos, repeticiones)"

conformidad["tipo"] = conformidad.ruta.map(clasificar)
resumen = conformidad.tipo.value_counts().sort_index()
ax = resumen.iloc[::-1].plot.barh(figsize=(10, 3), color="#1f6db0")
for i, v in enumerate(resumen.iloc[::-1]): ax.text(v, i, f"  {v:,} ({v / len(conformidad):.0%})".replace(",", "."), va="center")
ax.set_xlim(0, resumen.max() * 1.25); ax.set_ylabel(""); plt.title("Tipos de desvío respecto del procedimiento"); plt.tight_layout(); plt.show()

**Qué leer ahí.** La mayoría de los incidentes **no sigue el procedimiento al pie de la letra**, y el desvío más común es resolver sin pasar por «En curso». ¿Es un problema? **Depende:** puede ser trabajo legítimo (un incidente trivial que se resuelve al instante) o un paso que se está omitiendo. La conformidad **detecta** las diferencias; **juzgarlas** requiere conocer el negocio. Para ciberseguridad esto es una herramienta de auditoría: ¿se contuvo el incidente antes de cerrarlo? ¿Se escaló cuando correspondía?

**Un modelo del proceso real, sin que nadie lo escriba.** La otra mitad de PM4Py es **descubrir** el proceso a partir de los datos y medir qué tan bien lo representa:

In [ ]:
# «Inductive miner»: aprende un modelo del proceso desde los datos. noise_threshold=0.2 ignora el 20 % de comportamientos más raros.
modelo, ini, fn = pm4py.discover_petri_net_inductive(log, noise_threshold=0.2)
fitness = pm4py.fitness_token_based_replay(log, modelo, ini, fn)["average_trace_fitness"]
precision = pm4py.precision_token_based_replay(log, modelo, ini, fn)
ajuste_esperado = np.mean([d["trace_fitness"] for d in diagnostico])
print(f"Modelo descubierto de los datos: fitness {fitness:.2f} · precisión {precision:.2f}")
print(f"Procedimiento escrito a mano:    fitness {ajuste_esperado:.2f}")

**Dos medidas para evaluar un modelo de proceso:**
- ***Fitness* (ajuste):** de 0 a 1, qué parte de lo que realmente pasó **puede reproducir** el modelo. El procedimiento escrito explica bien, pero no todo (≈ 0,82); el modelo descubierto reproduce casi todo (0,99).
- **Precisión:** de 0 a 1, qué parte de lo que el modelo *permite* **ocurre de verdad**. Un modelo que permitiera cualquier cosa tendría fitness 1 pero precisión baja. El descubierto tiene 0,85.

Fitness y precisión **se contraponen**: un modelo muy general reproduce todo pero no dice nada; uno muy estrecho dice mucho pero no reproduce los casos raros. Es la misma tensión de siempre en ciencia de datos, ahora con procesos.

## 7 · ¿El camino se relaciona con cumplir el plazo?
Cada incidente tiene un **SLA** (acuerdo de nivel de servicio): un plazo para resolverlo. La columna `cumplio_sla` dice si se cumplió. ¿Los incidentes que siguen ciertos caminos incumplen más?

In [ ]:
sla = eventos.groupby("incidente").cumplio_sla.last()
tabla_sla = pd.DataFrame({"ruta": rutas, "incumplio_sla": ~sla.reindex(rutas.index)})
principales = variantes.head(7).index
t = tabla_sla[tabla_sla.ruta.isin(principales)].groupby("ruta").incumplio_sla.agg(incidentes="size", pct_incumple=lambda s: s.mean() * 100).sort_values("pct_incumple")
ax = t.pct_incumple.plot.barh(figsize=(10, 4), color=["#5cb85c" if v < 25 else "#f0ad4e" if v < 55 else "#d9534f" for v in t.pct_incumple])
for i, (v, n) in enumerate(zip(t.pct_incumple, t.incidentes)): ax.text(v + 1, i, f"{v:.0f} %  ({n:,} incidentes)".replace(",", "."), va="center", fontsize=9)
ax.axvline(tabla_sla.incumplio_sla.mean() * 100, color="gray", ls="--"); ax.set_xlim(0, 105); ax.set_ylabel("")
ax.set_yticklabels([r if len(r) < 52 else r[:49] + "…" for r in t.index], fontsize=8); ax.set_xlabel("% de incidentes que incumplen el plazo (línea gris: promedio general)")
plt.title("Incumplimiento del plazo según el camino (las 7 rutas más comunes)"); plt.tight_layout(); plt.show()

**Qué leer ahí.** Los incidentes que **esperan información del usuario** incumplen el plazo mucho más (entre 65 % y 75 %) que los que se resuelven sin esperas (entre 16 % y 22 %). El camino **se relaciona** con el cumplimiento. Pero ojo: eso **no demuestra que la espera cause el incumplimiento**; puede que los incidentes más difíciles necesiten más información *y* tarden más. La minería de procesos **muestra dónde mirar**, no por qué pasa.

## Cierre
- La **minería de procesos** reconstruye cómo ocurre un proceso a partir de **caso, actividad y fecha**.
- **Descubrir**: variantes y mapas; pocas rutas concentran casi todo.
- **Medir tiempos**: dónde esperan los incidentes (y cuáles esperas son fijas, no trabajo).
- **Comprobar conformidad**: qué incidentes se desviaron del procedimiento; **juzgar** el desvío exige conocer el negocio.
- Las variables que salen del proceso (cuántos eventos, cuántas esperas, cuánto tarda) pueden **alimentar modelos**: es lo que haremos en las dos prácticas siguientes, para **segmentar** los incidentes y **detectar los anómalos**.